# M5 Baseline LightGBM Pipeline on Cleaned Dataset

This notebook is a streamlined Colab-ready pipeline that:

- loads the pre-cleaned long-format dataset
- removes the old raw-data EDA and merge steps
- adds a minimal but strong feature set
- trains a LightGBM baseline model
- reports RMSE and MAE on the final 28-day validation window

https://github.com/Mcompetitions/M5-methods/tree/master/Code%20of%20Winning%20Methods


In [2]:
# =========================
# 1) Colab setup
# =========================
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
# =========================
# 2) Imports
# =========================
from pathlib import Path
import json
import gc

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, mean_absolute_error

pd.set_option("display.max_columns", 200)


In [4]:
# =========================
# 3) Configuration
# =========================
# Update these paths to your own Google Drive folders.
DATA_DIR = Path("/content/drive/MyDrive/Group Project - Predictive/data/cleaned_data")
CLEAN_DATA_PATH = DATA_DIR / "long_df_clean.parquet"

OUTPUT_DIR = Path("/content/drive/MyDrive/Group Project - Predictive/data/FE_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Fast iteration options
FAST_MODE = True
N_SERIES = 2000
RANDOM_STATE = 1234
VALID_DAYS = 28

# Experiment label
PIPELINE_NAME = "baseline_lgb_clean_minimal_features"
CURRENT_EXPERIMENT = "E0"

print("CLEAN_DATA_PATH:", CLEAN_DATA_PATH)
print("OUTPUT_DIR      :", OUTPUT_DIR)


CLEAN_DATA_PATH: /content/drive/MyDrive/Group Project - Predictive/data/cleaned_data/long_df_clean.parquet
OUTPUT_DIR      : /content/drive/MyDrive/Group Project - Predictive/data/FE_output


In [5]:
# =========================
# 4) Load cleaned dataset
# =========================
df = pd.read_parquet(CLEAN_DATA_PATH).copy()

print("Loaded shape:", df.shape)
print("Columns:", len(df.columns))
display(df.head(3))


Loaded shape: (16098720, 25)
Columns: 25


,id,item_id,dept_id,cat_id,store_id,state_id,d,demand,d_num,date,wm_yr_wk,wday,month,year,weekday,snap_CA,snap_TX,snap_WI,event_name_1,event_type_1,event_name_2,event_type_2,sell_price,is_active,snap_flag
0,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1414,1,1414,2014-12-12,11445,7,12,2014,Friday,0,1,1,NoEvent,NoEvent,NoEvent,NoEvent,2.24,1,0
1,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1415,2,1415,2014-12-13,11446,1,12,2014,Saturday,0,1,0,NoEvent,NoEvent,NoEvent,NoEvent,2.24,1,0
2,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1416,2,1416,2014-12-14,11446,2,12,2014,Sunday,0,0,1,NoEvent,NoEvent,NoEvent,NoEvent,2.24,1,0


In [6]:
# =========================
# 5) Basic type checks and optional fast-mode sampling
# =========================
df["date"] = pd.to_datetime(df["date"])

# Keep only the columns needed for this baseline.
required_base_cols = [
    "id", "item_id", "dept_id", "cat_id", "store_id", "state_id",
    "date", "demand", "sell_price"
]


# Optional fast-mode sampling at the series level.
if FAST_MODE:
    sampled_ids = (
        df["id"]
        .drop_duplicates()
        .sample(n=min(N_SERIES, df["id"].nunique()), random_state=RANDOM_STATE)
    )
    df = df[df["id"].isin(sampled_ids)].copy()

df = df.sort_values(["id", "date"]).reset_index(drop=True)

print("Working shape:", df.shape)
print("Unique series:", df["id"].nunique())
print("Date range   :", df["date"].min().date(), "to", df["date"].max().date())


Working shape: (1056000, 25)
Unique series: 2000
Date range   : 2014-12-12 to 2016-05-22


## Minimal feature set

This notebook uses a shared baseline feature family that is small enough for quick testing but still much stronger than a pure calendar-price baseline:

- calendar: `dow`, `month`
- price: `sell_price`, `price_change`, `is_promo`
- demand history: `lag_7`, `lag_28`, `rmean_7`, `rmean_28`


In [7]:
# =========================
# 6) Feature engineering
# =========================
df = df.sort_values(["id", "date"]).reset_index(drop=True)

# Calendar features
df["dow"] = df["date"].dt.dayofweek.astype("int8")
df["month"] = df["date"].dt.month.astype("int8")


# Price features
price_grp = df.groupby("id", sort=False)["sell_price"]
df["price_lag_1"] = price_grp.shift(1)
df["price_change"] = ((df["sell_price"] - df["price_lag_1"]) / df["price_lag_1"]).replace([np.inf, -np.inf], np.nan)
df["price_change"] = df["price_change"].fillna(0).astype("float32")
df["is_promo"] = (df["sell_price"] < df["price_lag_1"].fillna(df["sell_price"])).astype("int8")

# Demand history features
demand_grp = df.groupby("id", sort=False)["demand"]
df["lag_7"] = demand_grp.shift(7)
df["lag_28"] = demand_grp.shift(28)

df["rmean_7"] = (
    df.groupby("id", sort=False)["lag_7"]
      .transform(lambda x: x.rolling(7).mean())
)

df["rmean_28"] = (
    df.groupby("id", sort=False)["lag_28"]
      .transform(lambda x: x.rolling(28).mean())
)

# Downcast numeric features
for col in ["sell_price", "price_lag_1", "lag_7", "lag_28", "rmean_7", "rmean_28"]:
    df[col] = pd.to_numeric(df[col], downcast="float")

feature_cols = [
    "item_id", "dept_id", "cat_id", "store_id", "state_id",
    "dow", "month",
    "sell_price", "price_change", "is_promo",
    "lag_7", "lag_28", "rmean_7", "rmean_28"
]

required_history_cols = ["lag_28", "rmean_28"]
model_df = df.dropna(subset=required_history_cols).copy()

cat_cols = ["item_id", "dept_id", "cat_id", "store_id", "state_id"]
for col in cat_cols:
    model_df[col] = model_df[col].astype("category")

print("Model shape after dropping required history NaNs:", model_df.shape)
display(model_df[["id", "date"] + feature_cols + ["demand"]].head(3))

Model shape after dropping required history NaNs: (946000, 33)


,id,date,item_id,dept_id,cat_id,store_id,state_id,dow,month,sell_price,price_change,is_promo,lag_7,lag_28,rmean_7,rmean_28,demand
55,FOODS_1_001_WI_2_evaluation,2015-02-05,FOODS_1_001,FOODS_1,FOODS,WI_2,WI,3,2,2.24,0.0,0,1.0,0.0,0.571429,0.285714,0
56,FOODS_1_001_WI_2_evaluation,2015-02-06,FOODS_1_001,FOODS_1,FOODS,WI_2,WI,4,2,2.24,0.0,0,0.0,0.0,0.571429,0.214286,0
57,FOODS_1_001_WI_2_evaluation,2015-02-07,FOODS_1_001,FOODS_1,FOODS,WI_2,WI,5,2,2.24,0.0,0,0.0,0.0,0.428571,0.214286,2


In [8]:
# =========================
# Additional features for E1 ~ E4
# =========================

# Make sure the dataframe is sorted before using shift / rolling
df = df.sort_values(["id", "date"]).reset_index(drop=True)

demand_grp = df.groupby("id", sort=False)["demand"]

# -------------------------
# E1: Lag Expansion
# -------------------------
df["lag_14"] = demand_grp.shift(14)
df["lag_56"] = demand_grp.shift(56)

# -------------------------
# E2: Long-term Seasonality
# -------------------------
df["lag_365"] = demand_grp.shift(365)

# -------------------------
# E3: Rolling Enhancement
# -------------------------
df["rstd_28"] = (
    df.groupby("id", sort=False)["lag_28"]
      .transform(lambda x: x.rolling(28).std())
)

df["rmean_56"] = (
    demand_grp.shift(56)
    .groupby(df["id"], sort=False)
    .transform(lambda x: x.rolling(56).mean())
)

# -------------------------
# E4: Extreme Behavior
# -------------------------
df["rmax_28"] = (
    df.groupby("id", sort=False)["lag_28"]
      .transform(lambda x: x.rolling(28).max())
)

df["rmin_28"] = (
    df.groupby("id", sort=False)["lag_28"]
      .transform(lambda x: x.rolling(28).min())
)

# -------------------------
# Downcast new numeric features
# -------------------------
extra_num_cols = [
    "lag_14", "lag_56", "lag_365",
    "rstd_28", "rmean_56",
    "rmax_28", "rmin_28"
]

for col in extra_num_cols:
    df[col] = pd.to_numeric(df[col], downcast="float")

In [9]:
baseline_feature_cols = feature_cols.copy()
baseline_required_history_cols = required_history_cols.copy()

In [10]:
# Define experiment add-on features
experiment_addons = {
    "E0": [],
    "E1": ["lag_56"],
    "E2": ["lag_365"],
    "E3": ["rstd_28", "rmean_56"],
    "E4": ["rmax_28", "rmin_28"],
    "E10": ["lag_14", "lag_56", "rmax_28", "rmin_28"]
    #"E11": ["snap"]
    #"E12": ["snap", "lag_14", "lag_56", "rmax_28", "rmin_28"]

}

# Define which extra columns must be non-null for each experiment
experiment_required_addons = {
    "E0": [],
    "E1": ["lag_56"],
    "E2": ["lag_365"],
    "E3": ["rstd_28", "rmean_56"],
    "E4": ["rmax_28", "rmin_28"],
    "E10": ["lag_56", "rmax_28", "rmin_28"]
    # "E11": ["snap"],
    # "E12": ["snap", "lag_14", "lag_56", "rmax_28", "rmin_28"]
}

# para

In [11]:
# =========================
# Choose experiment here
# =========================
exp_name = "E0"   # change to E1 / E2 / E3 / E4 when needed

# Automatically build feature list and required history list
feature_cols = baseline_feature_cols + experiment_addons[exp_name]
required_history_cols = baseline_required_history_cols + experiment_required_addons[exp_name]

# Build model dataframe
model_df = df.dropna(subset=required_history_cols).copy()

# Convert categorical columns
cat_cols = ["item_id", "dept_id", "cat_id", "store_id", "state_id"]
for col in cat_cols:
    model_df[col] = model_df[col].astype("category")

print("Experiment:", exp_name)
print("Added features:", experiment_addons[exp_name] if experiment_addons[exp_name] else "None")
print("Number of features:", len(feature_cols))
print("Model shape:", model_df.shape)

display(model_df[["id", "date"] + feature_cols + ["demand"]].head(3))

Experiment: E0
Added features: None
Number of features: 14
Model shape: (946000, 40)


,id,date,item_id,dept_id,cat_id,store_id,state_id,dow,month,sell_price,price_change,is_promo,lag_7,lag_28,rmean_7,rmean_28,demand
55,FOODS_1_001_WI_2_evaluation,2015-02-05,FOODS_1_001,FOODS_1,FOODS,WI_2,WI,3,2,2.24,0.0,0,1.0,0.0,0.571429,0.285714,0
56,FOODS_1_001_WI_2_evaluation,2015-02-06,FOODS_1_001,FOODS_1,FOODS,WI_2,WI,4,2,2.24,0.0,0,0.0,0.0,0.571429,0.214286,0
57,FOODS_1_001_WI_2_evaluation,2015-02-07,FOODS_1_001,FOODS_1,FOODS,WI_2,WI,5,2,2.24,0.0,0,0.0,0.0,0.428571,0.214286,2


In [12]:
# =========================
# 7) Train / valid split
# =========================
max_date = model_df["date"].max()
valid_start = max_date - pd.Timedelta(days=VALID_DAYS - 1)

train_df = model_df[model_df["date"] < valid_start].copy()
valid_df = model_df[model_df["date"] >= valid_start].copy()

print("Validation start:", valid_start.date())
print("Validation end  :", max_date.date())
print("Train shape     :", train_df.shape)
print("Valid shape     :", valid_df.shape)
print("Train ids       :", train_df["id"].nunique())
print("Valid ids       :", valid_df["id"].nunique())


Validation start: 2016-04-25
Validation end  : 2016-05-22
Train shape     : (890000, 40)
Valid shape     : (56000, 40)
Train ids       : 2000
Valid ids       : 2000


In [13]:
# =========================
# 8) LightGBM configuration
# =========================
NUM_BOOST_ROUND = 2000
EARLY_STOPPING_ROUNDS = 100
VERBOSE_EVAL = 100

LGB_PARAMS = {
    "objective": "tweedie",
    "tweedie_variance_power": 1.5,
    "metric": "rmse",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 127,
    "min_data_in_leaf": 100,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l1": 0.0,
    "lambda_l2": 0.1,
    "seed": RANDOM_STATE,
    "verbosity": -1
}

print(json.dumps(LGB_PARAMS, indent=2))


{
  "objective": "tweedie",
  "tweedie_variance_power": 1.5,
  "metric": "rmse",
  "boosting_type": "gbdt",
  "learning_rate": 0.05,
  "num_leaves": 127,
  "min_data_in_leaf": 100,
  "feature_fraction": 0.8,
  "bagging_fraction": 0.8,
  "bagging_freq": 1,
  "lambda_l1": 0.0,
  "lambda_l2": 0.1,
  "seed": 1234,
  "verbosity": -1
}


In [14]:
# =========================
# 9) Prepare matrices
# =========================
X_train = train_df[feature_cols].copy()
y_train = train_df["demand"].copy()

X_valid = valid_df[feature_cols].copy()
y_valid = valid_df["demand"].copy()

categorical_cols_model = ["item_id", "dept_id", "cat_id", "store_id", "state_id"]

for col in categorical_cols_model:
    X_train[col] = X_train[col].astype("category")
    X_valid[col] = X_valid[col].astype("category")

train_set = lgb.Dataset(
    X_train,
    label=y_train,
    categorical_feature=categorical_cols_model,
    free_raw_data=False
)

valid_set = lgb.Dataset(
    X_valid,
    label=y_valid,
    categorical_feature=categorical_cols_model,
    free_raw_data=False
)


In [15]:
# =========================
# 10) Train model
# =========================
model = lgb.train(
    params=LGB_PARAMS,
    train_set=train_set,
    valid_sets=[train_set, valid_set],
    valid_names=["train", "valid"],
    num_boost_round=NUM_BOOST_ROUND,
    callbacks=[
        lgb.early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS),
        lgb.log_evaluation(period=VERBOSE_EVAL)
    ]
)


Training until validation scores don't improve for 100 rounds
[100]	train's rmse: 1.98521	valid's rmse: 2.16199
Early stopping, best iteration is:
[97]	train's rmse: 1.98788	valid's rmse: 2.16194


In [16]:
# =========================
# 11) Evaluate
# =========================
from sklearn.metrics import root_mean_squared_error, mean_absolute_error
valid_df["pred"] = model.predict(X_valid, num_iteration=model.best_iteration)

rmse = root_mean_squared_error(valid_df["demand"], valid_df["pred"])
mae = mean_absolute_error(valid_df["demand"], valid_df["pred"])

metrics_df = pd.DataFrame([{
    "pipeline_name": PIPELINE_NAME,
    "experiment_name": CURRENT_EXPERIMENT,
    "fast_mode": FAST_MODE,
    "n_series": int(df["id"].nunique()),
    "valid_days": VALID_DAYS,
    "rmse": rmse,
    "mae": mae,
    "best_iteration": int(model.best_iteration)
}])

print(metrics_df)


                         pipeline_name experiment_name  fast_mode  n_series  \
0  baseline_lgb_clean_minimal_features              E0       True      2000   

   valid_days      rmse       mae  best_iteration  
0          28  2.161936  1.016549              97  


# compare results

In [17]:
# =========================
# Save experiment result
# =========================

results1 = []
results1.append({
    "exp": exp_name,
    "added_features": ", ".join(experiment_addons[exp_name]) if experiment_addons[exp_name] else "None",
    "rmse": rmse,
    "mae": mae,
    "n_rows": model_df.shape[0],
    "n_features": len(feature_cols)
})

results_df1 = pd.DataFrame(results1)
display(results_df1)

,exp,added_features,rmse,mae,n_rows,n_features
0,E0,None,2.161936,1.016549,946000,14


In [18]:
# =========================
# LightGBM parameters
# =========================
NUM_BOOST_ROUND = 2000
EARLY_STOPPING_ROUNDS = 100
VERBOSE_EVAL = 100

LGB_PARAMS = {
    "objective": "tweedie",
    "tweedie_variance_power": 1.5,
    "metric": "rmse",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 127,
    "min_data_in_leaf": 100,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l1": 0.0,
    "lambda_l2": 0.1,
    "seed": RANDOM_STATE,
    "verbosity": -1
}

print(json.dumps(LGB_PARAMS, indent=2))

# =========================
# Auto-run experiments
# =========================

# Keep your original baseline definition
feature_cols = [
    "item_id", "dept_id", "cat_id", "store_id", "state_id",
    "dow", "month",
    "sell_price", "price_change", "is_promo",
    "lag_7", "lag_28", "rmean_7", "rmean_28"
]

required_history_cols = ["lag_28", "rmean_28"]

# Save baseline copies
baseline_feature_cols = feature_cols.copy()
baseline_required_history_cols = required_history_cols.copy()

# Define experiment add-on features
experiment_addons = {
    "E0": [],
    "E1": ["lag_56"],
    "E2": ["lag_365"],
    "E3": ["rstd_28", "rmean_56"],
    "E4": ["rmax_28", "rmin_28"],
    "E10": ["lag_56", "rmax_28", "rmin_28"],
}

# Define which extra columns must be non-null for each experiment
experiment_required_addons = {
    "E0": [],
    "E1": ["lag_56"],
    "E2": ["lag_365"],
    "E3": ["rstd_28", "rmean_56"],
    "E4": ["rmax_28", "rmin_28"],
    "E10": ["lag_56", "rmax_28", "rmin_28"],
}

# Reset results
results = []

# Optional: choose which experiments to run
experiments_to_run = ["E0", "E1", "E2", "E3", "E4", "E10"]

for exp_name in experiments_to_run:
    print("=" * 80)
    print("Running experiment:", exp_name)

    # Build feature set
    feature_cols = baseline_feature_cols + experiment_addons[exp_name]
    required_history_cols = baseline_required_history_cols + experiment_required_addons[exp_name]

    # Build model dataframe
    model_df = df.dropna(subset=required_history_cols).copy()

    # Convert categorical columns
    cat_cols = ["item_id", "dept_id", "cat_id", "store_id", "state_id"]
    for col in cat_cols:
        model_df[col] = model_df[col].astype("category")

    # Train / valid split
    max_date = model_df["date"].max()
    valid_start = max_date - pd.Timedelta(days=VALID_DAYS - 1)

    train_df = model_df[model_df["date"] < valid_start].copy()
    valid_df = model_df[model_df["date"] >= valid_start].copy()

    X_train = train_df[feature_cols]
    y_train = train_df["demand"]

    X_valid = valid_df[feature_cols]
    y_valid = valid_df["demand"]

    # LightGBM datasets
    lgb_train = lgb.Dataset(
        X_train,
        label=y_train,
        categorical_feature=cat_cols,
        free_raw_data=False
    )

    lgb_valid = lgb.Dataset(
        X_valid,
        label=y_valid,
        categorical_feature=cat_cols,
        free_raw_data=False
    )

    # Use your existing params here
    model = lgb.train(
        LGB_PARAMS,
        lgb_train,
        valid_sets=[lgb_train, lgb_valid],
        valid_names=["train", "valid"],
        num_boost_round=NUM_BOOST_ROUND,
        callbacks=[lgb.log_evaluation(100)]
    )

    # Predict
    valid_df["pred"] = model.predict(X_valid, num_iteration=model.best_iteration)

    # Metrics
    rmse = np.sqrt(mean_squared_error(valid_df["demand"], valid_df["pred"]))
    mae = mean_absolute_error(valid_df["demand"]
                              , valid_df["pred"])

    # Save result
    results.append({
        "exp": exp_name,
        "added_features": ", ".join(experiment_addons[exp_name]) if experiment_addons[exp_name] else "None",
        "rmse": rmse,
        "mae": mae,
        "n_rows": model_df.shape[0],
        "n_features": len(feature_cols)
    })

    print(f"Finished {exp_name}")
    print(f"RMSE: {rmse:.6f}")
    print(f"MAE : {mae:.6f}")
    print(f"Rows: {model_df.shape[0]}")
    print(f"Features used: {len(feature_cols)}")

results_df = pd.DataFrame(results).sort_values("rmse").reset_index(drop=True)
display(results_df)

{
  "objective": "tweedie",
  "tweedie_variance_power": 1.5,
  "metric": "rmse",
  "boosting_type": "gbdt",
  "learning_rate": 0.05,
  "num_leaves": 127,
  "min_data_in_leaf": 100,
  "feature_fraction": 0.8,
  "bagging_fraction": 0.8,
  "bagging_freq": 1,
  "lambda_l1": 0.0,
  "lambda_l2": 0.1,
  "seed": 1234,
  "verbosity": -1
}
Running experiment: E0
[100]	train's rmse: 1.98521	valid's rmse: 2.16199
[200]	train's rmse: 1.93942	valid's rmse: 2.17543
[300]	train's rmse: 1.91452	valid's rmse: 2.17889
[400]	train's rmse: 1.89071	valid's rmse: 2.18348
[500]	train's rmse: 1.87096	valid's rmse: 2.18619
[600]	train's rmse: 1.85636	valid's rmse: 2.18951
[700]	train's rmse: 1.84404	valid's rmse: 2.1927
[800]	train's rmse: 1.8308	valid's rmse: 2.1944
[900]	train's rmse: 1.81933	valid's rmse: 2.19972
[1000]	train's rmse: 1.80788	valid's rmse: 2.20453
[1100]	train's rmse: 1.79851	valid's rmse: 2.2056
[1200]	train's rmse: 1.78879	valid's rmse: 2.20716
[1300]	train's rmse: 1.77843	valid's rmse: 2.2

,exp,added_features,rmse,mae,n_rows,n_features
0,E1,lag_56,2.209559,1.009879,944000,15
1,E10,"lag_56, rmax_28, rmin_28",2.219352,1.011478,944000,17
2,E0,None,2.219788,1.013110,946000,14
3,E4,"rmax_28, rmin_28",2.220596,1.012336,946000,16
4,E3,"rstd_28, rmean_56",2.220834,1.008055,834000,16
5,E2,lag_365,2.287028,1.024269,326000,15


In [19]:
# =========================
# 12) Save outputs
# =========================
metrics_path = OUTPUT_DIR / f"{PIPELINE_NAME}_{CURRENT_EXPERIMENT}_metrics.csv"
preds_path   = OUTPUT_DIR / f"{PIPELINE_NAME}_{CURRENT_EXPERIMENT}_valid_predictions.csv"
fi_path      = OUTPUT_DIR / f"{PIPELINE_NAME}_{CURRENT_EXPERIMENT}_feature_importance.csv"

metrics_df.to_csv(metrics_path, index=False)

valid_df[["id", "date", "demand", "pred"]].to_csv(preds_path, index=False)

fi_df = pd.DataFrame({
     "feature": feature_cols,
     "importance_gain": model.feature_importance(importance_type="gain"),
     "importance_split": model.feature_importance(importance_type="split")
 }).sort_values("importance_gain", ascending=False)

fi_df.to_csv(fi_path, index=False)

print("Saved:")
print(metrics_path)
print(preds_path)
print(fi_path)

display(fi_df.head(15))


Saved:
/content/drive/MyDrive/Group Project - Predictive/data/FE_output/baseline_lgb_clean_minimal_features_E0_metrics.csv
/content/drive/MyDrive/Group Project - Predictive/data/FE_output/baseline_lgb_clean_minimal_features_E0_valid_predictions.csv
/content/drive/MyDrive/Group Project - Predictive/data/FE_output/baseline_lgb_clean_minimal_features_E0_feature_importance.csv


,feature,importance_gain,importance_split
12,rmean_7,9.829132e+06,20355
0,item_id,3.490718e+06,74236
13,rmean_28,2.010375e+06,31998
6,month,3.936401e+05,27709
15,rmax_28,3.392497e+05,12995
5,dow,2.860453e+05,17613
7,sell_price,2.779090e+05,17014
10,lag_7,1.996462e+05,7768
3,store_id,1.844487e+05,14675
11,lag_28,1.115702e+05,8307
